# Notebook 02: Cognitive Loop - วงจรการคิดของ Generative Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiraphap/Gen-Agent-Lectures-at-Econ-TU/blob/master/teaching/notebooks/02_cognitive_loop.ipynb)

## สิ่งที่จะได้เรียนรู้
1. **Cognitive Loop 5 ขั้นตอน**: Perceive → Retrieve → Plan → Execute → Reflect
2. **LLM เป็นสมองของ Agent**: ทำไมไม่ใช้ if-else แต่ใช้ LLM ตัดสินใจ
3. **Prompt Engineering**: วิธีสร้าง prompt ที่ดีให้ LLM
4. **Hands-on**: รัน cognitive loop จริง (ใช้ LLM หรือ mock ก็ได้)

---

## Background

ใน Notebook 01 เราเรียนรู้เรื่อง **Memory Stream** ไปแล้ว

ตอนนี้จะมาดูว่า Memory Stream ถูกใช้ใน **Cognitive Loop** อย่างไร

```
┌─────────────────────────────────────────────────────┐
│                  COGNITIVE LOOP                      │
│                                                      │
│   PERCEIVE → RETRIEVE → PLAN → EXECUTE → REFLECT   │
│   (สังเกต)   (จำ)      (คิด)  (ทำ)      (สรุป)    │
│                          ↑                    │      │
│                          │  LLM               │      │
│                          ↓                    ↓      │
│                    ┌──────────────┐                  │
│                    │Memory Stream │                  │
│                    └──────────────┘                  │
└─────────────────────────────────────────────────────┘
```

### ขั้นตอนไหนใช้ LLM?

| ขั้นตอน | ใช้ LLM? | อธิบาย |
|---------|----------|--------|
| 1. Perceive | ไม่ | แค่อ่านข้อมูลจาก environment |
| 2. Retrieve | ไม่ | ใช้สูตร recency × importance × relevance |
| 3. **Plan** | **ใช่** | **LLM คิดว่าจะทำอะไร** |
| 4. Execute | ไม่ | ลงมือทำตามแผน |
| 5. **Reflect** | **ใช่** | **LLM สรุปบทเรียนจากประสบการณ์** |

---
**ผศ.ดร.ถิรภาพ ฟักทอง** | คณะเศรษฐศาสตร์ มหาวิทยาลัยธรรมศาสตร์

In [ ]:
# ============================================================
# CONFIG: เลือกว่าจะใช้ LLM จริงหรือ mock responses
# ============================================================

USE_REAL_LLM = False  # ← เปลี่ยนเป็น True ถ้ามี OpenAI API key

# ถ้าใช้ LLM จริง ต้อง install และตั้งค่า:
# !pip install openai python-dotenv
# และตั้ง environment variable: OPENAI_API_KEY=sk-...

if USE_REAL_LLM:
    try:
        import os
        from openai import OpenAI
        client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
        print(f"ใช้ LLM จริง: {MODEL}")
    except Exception as e:
        print(f"ไม่สามารถเชื่อมต่อ LLM ได้: {e}")
        print("สลับไปใช้ mock responses แทน")
        USE_REAL_LLM = False

if not USE_REAL_LLM:
    print("ใช้ Mock Responses (ไม่ต้องมี API key)")
    print("→ ถ้าอยากลอง LLM จริง เปลี่ยน USE_REAL_LLM = True")

In [ ]:
import json
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Optional

# ============================================================
# Memory class (จาก Notebook 01)
# ============================================================

@dataclass
class Memory:
    content: str
    created_at: datetime
    importance: int
    memory_type: str = "observation"
    related_store: Optional[str] = None


class MemoryStream:
    RECENCY_DECAY = 0.99
    
    def __init__(self, agent_name: str):
        self.agent_name = agent_name
        self.memories: list[Memory] = []
    
    def add(self, content, importance, memory_type="observation",
            created_at=None, related_store=None):
        mem = Memory(content=content, created_at=created_at or datetime.now(),
                     importance=importance, memory_type=memory_type,
                     related_store=related_store)
        self.memories.append(mem)
        return mem
    
    def retrieve(self, query, top_k=5, current_time=None):
        if current_time is None:
            current_time = datetime.now()
        results = []
        for mem in self.memories:
            hours = (current_time - mem.created_at).total_seconds() / 3600
            recency = self.RECENCY_DECAY ** hours
            importance = mem.importance / 10.0
            # substring matching (รองรับภาษาไทยที่ไม่มีช่องว่างระหว่างคำ)
            query_keywords = set(query.lower().split())
            content_lower = mem.content.lower()
            if not query_keywords:
                relevance = 0
            else:
                matching = sum(1 for word in query_keywords if word in content_lower)
                relevance = min(1.0, matching / len(query_keywords))
            total = recency * importance * relevance
            results.append((mem, total))
        results.sort(key=lambda x: x[1], reverse=True)
        return results[:top_k]
    
    def format_for_prompt(self, results):
        if not results:
            return "ไม่มีความทรงจำที่เกี่ยวข้อง"
        return "\n".join(f"- {mem.content}" for mem, _ in results)


print("Memory classes พร้อมใช้งาน!")

In [ ]:
# ============================================================
# Mock LLM Responses (ใช้เมื่อ USE_REAL_LLM = False)
# ============================================================
#
# Pre-recorded responses ที่เหมือนจริง
# เวลาสอน สามารถดู responses เหล่านี้เพื่อเข้าใจว่า LLM ตอบอะไร

MOCK_PLAN_RESPONSE = {
    "thinking_steps": [
        "วันนี้วันเสาร์ ไม่รีบ เป็นวันที่ชอบไปซื้อของอยู่แล้ว",
        "เช็คของที่ต้องซื้อ: นมเหลือ 1 กล่อง (เร่งด่วน), ไข่หมด (เร่งด่วน), ผงซักฟอกเหลือน้อย",
        "จำได้ว่าเห็นใบปลิว Big C ลดนม 30% - เป็นคนชอบดูโปรอยู่แล้ว (promotion_attraction = 9/10)",
        "แต่จำได้ว่าเมื่อสัปดาห์ก่อนไป Big C วันเสาร์ คนเยอะมาก ต่อคิวนาน ไม่ชอบเลย",
        "Lotus's บริการดีมาก พนักงานช่วยหาของ แต่ไม่มีโปรนมตอนนี้",
        "เมื่อชั่งน้ำหนัก: โปรนม 30% ที่ Big C คุ้มค่ามาก ประหยัดได้เยอะ (price_sensitivity = 8/10)",
        "ตัดสินใจไป Big C เพราะโปรนม 30% คุ้มค่า ยอมทนคนเยอะได้ เพราะประหยัดเงินได้เยอะกว่า"
    ],
    "decision": {
        "action": "go_shopping",
        "store": "Big C",
        "items_to_buy": ["นม Dutch Mill (โปร 30%)", "ไข่ 1 แผง", "ผงซักฟอก", "มะละกอ"],
        "estimated_spend": 850,
        "reasoning": "เลือก Big C เพราะมีโปรลดนม 30% ตรงกับที่ต้องซื้อ แม้จะคนเยอะแต่ประหยัดได้คุ้ม และซื้อของอื่นที่ต้องการได้ครบในที่เดียว"
    }
}

MOCK_REFLECT_RESPONSE = {
    "new_memory": "ไป Big C วันเสาร์ ได้นมลด 30% คุ้มมาก แต่คนเยอะเหมือนเดิม ต้องไปแต่เช้าครั้งหน้า",
    "importance": 7
}

print("Mock responses พร้อมใช้งาน!")
print(f"  - Plan response: {len(MOCK_PLAN_RESPONSE['thinking_steps'])} thinking steps")
print(f"  - Reflect response: importance = {MOCK_REFLECT_RESPONSE['importance']}/10")

In [ ]:
# ============================================================
# Customer Profile: สมศรี
# ============================================================
#
# ข้อมูลจาก data/customers.json ของ project

AGENT = {
    "name": "สมศรี",
    "age": 45,
    "job": "แม่บ้าน",
    "financial": {
        "monthly_income": 35000,
        "living_situation": "พอมีพอใช้ เก็บออมได้บ้าง",
        "financial_goal": "เก็บเงินให้ลูกเรียนมหาลัย"
    },
    "household": {
        "size": 4,
        "type": "บ้านเดี่ยว อยู่กับสามีและลูก 2 คน"
    },
    "personality": {
        "price_sensitivity": 0.8,
        "brand_loyalty": 0.6,
        "quality_preference": 0.5,
        "convenience_preference": 0.4,
        "promotion_attraction": 0.9,
        "impulse_buying": 0.3
    },
    "innate_traits": ["ใจเย็น", "ละเอียดรอบคอบ", "ชอบเปรียบเทียบราคา"],
    "interests": {
        "hobbies": ["ทำอาหาร", "ดูละคร", "ปลูกต้นไม้"],
        "favorite_categories": ["food", "household", "fresh_produce"]
    },
    "routines": [
        "ดูใบปลิวโปรโมชั่นทุกสัปดาห์",
        "ไปซื้อของสดตลาดเช้าวันอาทิตย์",
        "ทำกับข้าวให้ครอบครัวทุกวัน"
    ],
    "pain_points": [
        "ไม่ชอบร้านที่จอดรถยาก",
        "หงุดหงิดถ้าของที่ต้องการหมด",
        "ไม่ชอบพนักงานไม่สุภาพ"
    ],
    "background": "แม่บ้านที่ดูแลครอบครัว 4 คน ชอบดูใบปลิวโปรโมชั่นทุกสัปดาห์ ละเอียดรอบคอบเรื่องราคา"
}

# ร้านค้าที่มีให้เลือก
STORES = [
    {"name": "CJ More", "type": "supermarket", "promos": ["ลดผงซักฟอก 15%"]},
    {"name": "Big C", "type": "hypermarket", "promos": ["ลดนม Dutch Mill 30%", "ลดไข่ 10%"]},
    {"name": "Lotus's", "type": "hypermarket", "promos": ["ลดน้ำมันพืช 20%"]},
    {"name": "7-Eleven", "type": "convenience", "promos": ["กาแฟลด 50% เช้า"]},
    {"name": "Makro", "type": "wholesale", "promos": ["ซื้อ 3 แถม 1 สินค้า household"]}
]

print(f"Agent: {AGENT['name']} ({AGENT['job']}, อายุ {AGENT['age']})")
print(f"ครอบครัว: {AGENT['household']['size']} คน")
print(f"ร้านค้า: {len(STORES)} ร้าน")

## Step 1: PERCEIVE (สังเกต)

Agent สังเกตสิ่งรอบตัว:
- ตอนนี้วันอะไร กี่โมง?
- ของในตู้เย็น/บ้านเหลือแค่ไหน?
- มีร้านค้าไหนบ้าง มีโปรอะไร?

ขั้นตอนนี้ **ไม่ใช้ LLM** - แค่รวบรวมข้อมูลจาก environment

In [ ]:
# ============================================================
# STEP 1: PERCEIVE - สังเกตสิ่งรอบตัว
# ============================================================

NOW = datetime(2024, 1, 13, 10, 0)  # วันเสาร์ 10:00 น.

def perceive(agent, stores, current_time):
    """
    Agent สังเกตสิ่งรอบตัว
    
    ขั้นตอนนี้ไม่ใช้ LLM - แค่รวบรวมข้อมูล
    
    Returns:
        dict ข้อมูลที่ agent สังเกตเห็น
    """
    # ข้อมูลสถานการณ์
    day_names = {0: "จันทร์", 1: "อังคาร", 2: "พุธ", 3: "พฤหัส", 
                 4: "ศุกร์", 5: "เสาร์", 6: "อาทิตย์"}
    
    perception = {
        "day": day_names[current_time.weekday()],
        "time": current_time.strftime("%H:%M"),
        "date": current_time.strftime("%Y-%m-%d"),
        
        # ของที่ต้องซื้อ (สมมติ)
        "shopping_list": [
            {"item": "นม", "urgency": "high", "reason": "เหลือ 1 กล่อง"},
            {"item": "ไข่", "urgency": "high", "reason": "หมดแล้ว"},
            {"item": "ผงซักฟอก", "urgency": "medium", "reason": "เหลือน้อย"},
            {"item": "มะละกอ", "urgency": "low", "reason": "สามีอยากกินส้มตำ"},
        ],
        "budget": 3000,
        "mood": "สบายๆ ไม่รีบ",
        
        # ร้านค้าและโปรโมชั่น
        "available_stores": stores,
        
        # Query สำหรับ retrieve memories
        "query": "ซื้อของ ร้าน นม ไข่ โปรโมชั่น ลดราคา"
    }
    return perception


# รัน Perceive
print("╔" + "═" * 58 + "╗")
print("║  STEP 1: PERCEIVE (สังเกต)                               ║")
print("║  ไม่ใช้ LLM - แค่รวบรวมข้อมูลจาก environment             ║")
print("╚" + "═" * 58 + "╝")

perception = perceive(AGENT, STORES, NOW)

print(f"\n  วัน: {perception['day']}")
print(f"  เวลา: {perception['time']}")
print(f"  อารมณ์: {perception['mood']}")
print(f"  งบ: {perception['budget']:,} บาท")
print(f"\n  ของที่ต้องซื้อ:")
for item in perception['shopping_list']:
    urgency_icon = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(item['urgency'], "⚪")
    print(f"    {urgency_icon} {item['item']} - {item['reason']} [{item['urgency']}]")

print(f"\n  ร้านค้าที่มี:")
for store in perception['available_stores']:
    promos = ", ".join(store['promos'])
    print(f"    - {store['name']} ({store['type']}) | โปร: {promos}")

## Step 2: RETRIEVE (ดึงความทรงจำ)

Agent ดึง memories ที่เกี่ยวข้องกับสถานการณ์ปัจจุบัน

ใช้สูตร: `score = recency × importance × relevance`

ขั้นตอนนี้ **ไม่ใช้ LLM** - ใช้สูตรคำนวณอย่างเดียว

In [ ]:
# ============================================================
# STEP 2: RETRIEVE - ดึงความทรงจำที่เกี่ยวข้อง
# ============================================================

# สร้าง MemoryStream ให้สมศรี (เหมือน Notebook 01)
memory_stream = MemoryStream(agent_name="สมศรี")

# เพิ่ม memories
memories_data = [
    ("ไป Lotus's ซื้อของ พนักงานช่วยหาของดีมาก บริการประทับใจ", 7, "observation", 14*24, "Lotus's"),
    ("ซื้อของที่ Makro ราคาถูกแต่ต้องซื้อเยอะ เก็บไม่หมด", 5, "observation", 13*24, "Makro"),
    ("Lotus's มีโปรลดนม Dutch Mill 20% ซื้อมา 2 กล่อง", 6, "observation", 7*24, "Lotus's"),
    ("ไป Big C วันเสาร์ คนเยอะมาก ต่อคิวนาน ไม่ชอบเลย", 7, "observation", 7*24+2, "Big C"),
    ("Big C มีของครบกว่า Lotus's โดยเฉพาะผงซักฟอก หลายยี่ห้อ", 5, "reflection", 6*24, "Big C"),
    ("ทำอาหารเย็นให้ครอบครัว ใช้ไข่หมด 10 ฟอง", 4, "observation", 5*24, None),
    ("ลูกบอกว่าอยากกินนม Dutch Mill สตรอเบอร์รี่", 5, "observation", 4*24, None),
    ("ดูใบปลิว Big C เห็นโปรลดนม 30% สัปดาห์นี้", 7, "observation", 3*24, "Big C"),
    ("ผงซักฟอกเหลือน้อยแล้ว ต้องซื้อเพิ่ม", 6, "observation", 3*24, None),
    ("เช็คตู้เย็น นมเหลือ 1 กล่อง ต้องซื้อเพิ่มแน่ๆ", 8, "observation", 24, None),
    ("ไข่หมดไปเมื่อวาน ต้องซื้อด้วย", 7, "observation", 26, None),
    ("CJ More ใกล้บ้านที่สุด เดิน 5 นาทีถึง แต่ของไม่ค่อยครบ", 5, "reflection", 20, "CJ More"),
    ("วันนี้วันเสาร์ ไม่รีบ อากาศดี น่าออกไปซื้อของ", 4, "observation", 2, None),
    ("สามีบอกว่าอยากกินส้มตำ ต้องซื้อมะละกอด้วย", 5, "observation", 1, None),
    ("ควรไปซื้อของวันนี้ เพราะนมกับไข่เหลือน้อย", 8, "plan", 0.5, None),
]

for content, imp, mtype, hours_ago, store in memories_data:
    memory_stream.add(content, imp, mtype, 
                      created_at=NOW - timedelta(hours=hours_ago),
                      related_store=store)

# Retrieve!
query = perception['query']
retrieved = memory_stream.retrieve(query, top_k=5, current_time=NOW)

print("╔" + "═" * 58 + "╗")
print("║  STEP 2: RETRIEVE (ดึงความทรงจำ)                         ║")
print("║  ไม่ใช้ LLM - ใช้สูตร recency × importance × relevance   ║")
print("╚" + "═" * 58 + "╝")

print(f"\n  Query: \"{query}\"")
print(f"  จาก {len(memory_stream.memories)} memories → เลือก top {len(retrieved)}")
print()

for i, (mem, score) in enumerate(retrieved, 1):
    bar = "█" * int(score * 40)
    print(f"  #{i} [score={score:.4f}] {mem.content}")
    print(f"     {bar}")

# เก็บ formatted memories สำหรับ prompt
memories_text = memory_stream.format_for_prompt(retrieved)
print(f"\n  → memories เหล่านี้จะถูกส่งเข้า LLM ในขั้นตอน Plan")

## Step 3: PLAN (วางแผน) - ใช้ LLM!

นี่คือขั้นตอนที่ **LLM เป็นสมอง** ของ agent!

เราจะสร้าง prompt ที่ประกอบด้วย:
1. **ตัวตน** ของ agent (ชื่อ อายุ อาชีพ ครอบครัว)
2. **การเงิน** (รายได้ เป้าหมาย)
3. **บุคลิก** (personality traits 6 ด้าน)
4. **ความทรงจำ** ที่เกี่ยวข้อง (จาก Step 2)
5. **สถานการณ์** ปัจจุบัน
6. **คำถาม**: "จะทำอะไรดี?"

LLM จะ **คิดทีละขั้นตอน** (step-by-step reasoning) แล้วตัดสินใจ

In [ ]:
# ============================================================
# STEP 3: PLAN - ใช้ LLM ตัดสินใจ!
# ============================================================

def build_prompt(agent, perception, memories_text):
    """
    สร้าง prompt สำหรับ LLM
    
    Prompt ที่ดีต้องให้ข้อมูลครบ เพื่อให้ LLM ตัดสินใจ "ในบทบาท" ของ agent
    """
    p = agent['personality']
    fin = agent['financial']
    hh = agent['household']
    
    # Format stores
    stores_text = []
    for store in perception['available_stores']:
        promos = ", ".join(store['promos'])
        stores_text.append(f"- {store['name']} ({store['type']}) - โปร: {promos}")
    
    # Format shopping list
    shopping_text = []
    for item in perception['shopping_list']:
        shopping_text.append(f"- {item['item']} ({item['reason']}) [ความเร่งด่วน: {item['urgency']}]")
    
    prompt = f"""คุณกำลังเล่นเป็นตัวละครชื่อ \"{agent['name']}\"

[ตัวตนของฉัน]
- อายุ {agent['age']} ปี อาชีพ {agent['job']}
- รายได้ครอบครัว {fin['monthly_income']:,} บาท/เดือน
- สถานะการเงิน: {fin['living_situation']}
- เป้าหมายการเงิน: {fin['financial_goal']}
- มีสมาชิกในบ้าน {hh['size']} คน ({hh['type']})
- {agent['background']}

[นิสัยของฉัน (personality traits)]
- ความไวต่อราคา: {p['price_sensitivity']*10:.0f}/10 (ยิ่งสูง = ดูราคามาก)
- ความภักดีต่อแบรนด์: {p['brand_loyalty']*10:.0f}/10
- ชอบของดี: {p['quality_preference']*10:.0f}/10
- ต้องการความสะดวก: {p['convenience_preference']*10:.0f}/10 (ยิ่งต่ำ = ยอมไปไกลได้)
- ชอบโปรโมชั่น: {p['promotion_attraction']*10:.0f}/10 (ยิ่งสูง = หลงโปรง่าย)
- ซื้อของตามใจ: {p['impulse_buying']*10:.0f}/10

[ลักษณะนิสัยโดยกำเนิด]
{chr(10).join(f'- {t}' for t in agent.get('innate_traits', []))}

[กิจวัตรประจำ]
{chr(10).join(f'- {r}' for r in agent.get('routines', []))}

[สิ่งที่ไม่ชอบ]
{chr(10).join(f'- {pp}' for pp in agent.get('pain_points', []))}

[ความทรงจำที่เกี่ยวข้อง]
{memories_text}

[สถานการณ์ตอนนี้]
- วัน{perception['day']} เวลา {perception['time']} ({perception['mood']})
- งบประมาณ: {perception['budget']:,} บาท

[ของที่ต้องซื้อ]
{chr(10).join(shopping_text)}

[ร้านค้าที่เลือกได้]
{chr(10).join(stores_text)}

---

จากข้อมูลทั้งหมด ให้ตัดสินใจว่าจะไปซื้อของที่ไหน
**คิดทีละขั้นตอนตามนิสัยของตัวละคร** แล้วสรุปการตัดสินใจ

ตอบในรูปแบบ JSON:
{{
    "thinking_steps": ["ความคิดที่ 1", "ความคิดที่ 2", ...],
    "decision": {{
        "action": "go_shopping" หรือ "stay_home",
        "store": "ชื่อร้าน",
        "items_to_buy": ["รายการ1", "รายการ2"],
        "estimated_spend": จำนวนเงินที่คาดว่าจะใช้,
        "reasoning": "สรุปเหตุผลสั้นๆ"
    }}
}}"""
    return prompt


# สร้าง prompt
prompt = build_prompt(AGENT, perception, memories_text)

print("╔" + "═" * 58 + "╗")
print("║  STEP 3: PLAN (วางแผน) - ใช้ LLM!                        ║")
print("╚" + "═" * 58 + "╝")

print("\n" + "─" * 60)
print("PROMPT ที่ส่งให้ LLM (ดูทั้งหมดเพื่อเข้าใจว่า agent รู้อะไรบ้าง):")
print("─" * 60)
print(prompt)
print("─" * 60)
print(f"\nPrompt length: {len(prompt)} characters")

In [ ]:
# ============================================================
# เรียก LLM (หรือใช้ mock response)
# ============================================================

def call_llm(prompt, system_message="คุณกำลังเล่นเป็นตัวละคร ตัดสินใจตามนิสัย ตอบเป็น JSON"):
    """เรียก LLM จริง หรือใช้ mock response"""
    if USE_REAL_LLM:
        print("  → กำลังส่ง prompt ไป LLM...")
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=1000,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)
    else:
        print("  → ใช้ Mock Response (ไม่ได้เรียก LLM จริง)")
        return None  # จะใช้ mock data ข้างนอก


# เรียก LLM
result = call_llm(prompt)
plan = result if result else MOCK_PLAN_RESPONSE

# แสดงผล
print("\n" + "─" * 60)
print("LLM RESPONSE - กระบวนการคิดของ สมศรี:")
print("─" * 60)

print("\n  Thinking Steps (คิดทีละขั้น):")
for i, step in enumerate(plan['thinking_steps'], 1):
    print(f"    {i}. {step}")

d = plan['decision']
print(f"\n  การตัดสินใจ:")
print(f"    Action: {d['action']}")
print(f"    ร้าน: {d.get('store', 'N/A')}")
print(f"    ของที่จะซื้อ: {', '.join(d.get('items_to_buy', []))}")
print(f"    งบประมาณ: {d.get('estimated_spend', 'N/A')} บาท")
print(f"    เหตุผล: {d['reasoning']}")

print("\n  สังเกต: สมศรีคิดตาม personality traits ของเธอ!")
print(f"    - price_sensitivity = {AGENT['personality']['price_sensitivity']} → ดูราคาเป็นหลัก")
print(f"    - promotion_attraction = {AGENT['personality']['promotion_attraction']} → หลงโปรง่าย")

## Step 4: EXECUTE (ลงมือทำ)

Agent ลงมือทำตามแผน:
- เดินทางไปร้านที่เลือก
- ซื้อของตาม shopping list
- หักงบประมาณ

ขั้นตอนนี้ **ไม่ใช้ LLM** - แค่ update state

In [ ]:
# ============================================================
# STEP 4: EXECUTE - ลงมือทำตามแผน
# ============================================================

print("╔" + "═" * 58 + "╗")
print("║  STEP 4: EXECUTE (ลงมือทำ)                                ║")
print("║  ไม่ใช้ LLM - update state ตามแผน                        ║")
print("╚" + "═" * 58 + "╝")

d = plan['decision']

if d['action'] == 'go_shopping':
    store = d.get('store', 'Unknown')
    items = d.get('items_to_buy', [])
    spend = d.get('estimated_spend', 0)
    
    print(f"\n  สมศรี กำลังเดินทางไป {store}...")
    print(f"  ถึงร้านแล้ว! กำลังซื้อของ:")
    for item in items:
        print(f"    ✓ {item}")
    print(f"\n  จ่ายเงิน: {spend:,} บาท")
    print(f"  งบเหลือ: {perception['budget'] - spend:,} บาท")
    
    # สร้าง transaction record
    transaction = {
        "agent": AGENT['name'],
        "store": store,
        "items": items,
        "amount": spend,
        "timestamp": NOW.isoformat(),
        "reasoning": d['reasoning']
    }
    print(f"\n  Transaction บันทึกแล้ว!")
else:
    print(f"\n  สมศรี ตัดสินใจอยู่บ้านวันนี้")
    print(f"  เหตุผล: {d['reasoning']}")

## Step 5: REFLECT (สรุปบทเรียน) - ใช้ LLM!

หลังจากทำกิจกรรมเสร็จ agent จะ **สรุปบทเรียน** จากประสบการณ์:
- สร้าง "ความทรงจำใหม่" (reflection)
- ให้คะแนนความสำคัญ 1-10
- เก็บเข้า Memory Stream เพื่อใช้ตัดสินใจในอนาคต

นี่คือ **วงจรที่สมบูรณ์**: ประสบการณ์ → ความทรงจำ → การตัดสินใจ → ประสบการณ์ใหม่ → ...

In [ ]:
# ============================================================
# STEP 5: REFLECT - สรุปบทเรียน ใช้ LLM!
# ============================================================

def build_reflect_prompt(agent, decision):
    """สร้าง prompt สำหรับ reflection"""
    return f"""คุณคือ \"{agent['name']}\" เพิ่งไปซื้อของ

สิ่งที่ทำ: {decision['reasoning']}
ร้าน: {decision.get('store', 'N/A')}
ของที่ซื้อ: {', '.join(decision.get('items_to_buy', []))}

จากประสบการณ์นี้ สร้าง "ความทรงจำใหม่" สั้นๆ 1-2 ประโยค
และให้คะแนนความสำคัญ 1-10 (10 = จะจำไปนานมาก จะเปลี่ยนพฤติกรรมเลย)

ตอบเป็น JSON:
{{
    "new_memory": "ประโยคความทรงจำใหม่",
    "importance": 1-10
}}"""


print("╔" + "═" * 58 + "╗")
print("║  STEP 5: REFLECT (สรุปบทเรียน) - ใช้ LLM!               ║")
print("╚" + "═" * 58 + "╝")

reflect_prompt = build_reflect_prompt(AGENT, plan['decision'])
print(f"\n  Prompt สำหรับ reflection:")
print(f"  {'-'*50}")
print(f"  {reflect_prompt}")
print(f"  {'-'*50}")

# เรียก LLM
reflect_result = call_llm(reflect_prompt, "สร้างความทรงจำสั้นๆ จากประสบการณ์ ตอบเป็น JSON")
reflection = reflect_result if reflect_result else MOCK_REFLECT_RESPONSE

print(f"\n  New Memory: \"{reflection['new_memory']}\"")
print(f"  Importance: {reflection['importance']}/10")

# เพิ่มเข้า Memory Stream!
memory_stream.add(
    content=reflection['new_memory'],
    importance=reflection['importance'],
    memory_type="reflection",
    created_at=NOW + timedelta(hours=1),  # หลังจากซื้อของเสร็จ
    related_store=plan['decision'].get('store')
)

print(f"\n  → เก็บเข้า Memory Stream แล้ว!")
print(f"  → ตอนนี้สมศรีมี {len(memory_stream.memories)} memories")
print(f"\n  ครั้งหน้าที่สมศรีต้องตัดสินใจ memory นี้จะถูกดึงมาใช้!")

## สรุป: Cognitive Loop ครบ 5 ขั้นตอน!

```
1. PERCEIVE  → สังเกตสิ่งรอบตัว (ไม่ใช้ LLM)
       ↓
2. RETRIEVE  → ดึงความทรงจำที่เกี่ยวข้อง (ไม่ใช้ LLM, ใช้สูตร)
       ↓
3. PLAN      → LLM คิดทีละขั้นตอน แล้วตัดสินใจ (ใช้ LLM!)
       ↓
4. EXECUTE   → ลงมือทำตามแผน (ไม่ใช้ LLM)
       ↓
5. REFLECT   → LLM สรุปบทเรียน เก็บเข้า Memory (ใช้ LLM!)
       ↓
    กลับไป 1. PERCEIVE (วนลูป)
```

### Key Insights

1. **LLM = สมอง** ของ agent ใช้ในขั้น Plan และ Reflect
2. **Memory Stream = ความจำ** เก็บทุกประสบการณ์ ดึงมาใช้ตัดสินใจ
3. **Personality Traits** ถูกฝังใน prompt → LLM ตัดสินใจ "ตามนิสัย"
4. **ไม่มี if-else!** ทุกการตัดสินใจผ่าน LLM = realistic มากกว่า rule-based

### Next Step
→ ไป **Notebook 03: Retail Simulation** เพื่อรัน simulation หลาย agents หลายวัน!

In [ ]:
# ============================================================
# แบบฝึกหัด
# ============================================================

# 1. ลองเปลี่ยน personality ของสมศรี แล้วดูว่า LLM ตัดสินใจต่างกันไหม
#    เช่น: เปลี่ยน price_sensitivity จาก 0.8 เป็น 0.2
#    แล้ว Run Step 3 ใหม่

# 2. ลองเปลี่ยน shopping list - ถ้ามีแค่ "กาแฟ" อย่างเดียว
#    สมศรีจะตัดสินใจไปร้านไหน?

# 3. ลองเพิ่ม memory ใหม่ที่บอกว่า "Big C ปิดปรับปรุงสัปดาห์นี้"
#    แล้ว Run Step 2-3 ใหม่ ดูว่าการตัดสินใจเปลี่ยนไหม

print("ลองทำแบบฝึกหัดข้างบนดูนะ!")
print("Hint: เปลี่ยน AGENT['personality'] dict แล้ว re-run cells")